# DPR (cosDPR-distil) pipeline on Colab

Builds the DPR Stage-1 index over MS MARCO (8.8M passages), verifies it against
Pyserini's published baselines, and produces the DPR → MiniLM score matrices.

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Google Drive needs **~14 GB free** for the embedding shards (float16, 89 shards of 100k passages).
- Steps 7–8 read the index from **local disk** (step 6b): Colab's Drive mount drops out (`Errno 107 Transport endpoint is not connected`) on long reads of the 13.5 GB index.
- Encoding is **resumable**: if the session disconnects, rerun the setup cells (1–4) and then the encode cell; finished shards are skipped.

| Step | Cell | Rough time |
|---|---|---|
| Setup + corpus download | 1–4 | ~10 min |
| Encoder fidelity check | 5 | ~2 min |
| Encode corpus | 6 | a few hours (the log prints passages/s) |
| Copy index to local disk | 6b | ~5–10 min |
| Verify Stage 1 vs published | 7 | ~5 min |
| Score matrices (DL19, DL20) | 8 | ~15 min |

## 1. GPU and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/IRE_PROJECT'
INDEX_DIR  = f'{DRIVE_ROOT}/dpr_index_cosdpr_distil'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
import os
os.makedirs(INDEX_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ['DPR_INDEX_DIR'] = INDEX_DIR
!df -h /content/drive | tail -1

## 2. Code

In [ ]:
BRANCH = 'dpr-pipeline'   # switch to 'main' once the DPR code is merged
!rm -rf /content/IRE_PROJECT
!git clone -q -b {BRANCH} https://github.com/a-n-u-p-a-m/IRE_PROJECT.git /content/IRE_PROJECT
%cd /content/IRE_PROJECT
!git log --oneline -1

## 3. Dependencies

Colab already ships torch, transformers and sentence-transformers. We use those
rather than reinstalling the pinned stack (which would replace Colab's CUDA torch
and force a restart); cell 5 checks the encoder output against Castorini's
reference ONNX model, so library-version drift can't silently change embeddings.

In [ ]:
!pip install -q onnxruntime pyyaml
import torch, transformers, sentence_transformers, numpy
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| sentence-transformers', sentence_transformers.__version__, '| numpy', numpy.__version__)

## 4. MS MARCO corpus (to local disk, ~4 GB)

In [ ]:
!python -c "from shared.data import _ensure_collection_tsv; print(_ensure_collection_tsv())"
!wc -l data/msmarco/collection.tsv   # expect 8841823

## 5. Encoder fidelity check

Compares our encoder (with the fp16 autocast used for the corpus) against
Castorini's ONNX export of cosDPR-distil, which Pyserini/Anserini use to reproduce
the published numbers. Expect cosine ≥ 0.999 for every text.

In [ ]:
import numpy as np, onnxruntime as ort
from huggingface_hub import hf_hub_download
from retrievers.dpr import CosDPREncoder
from shared.data import load_queries

enc = CosDPREncoder(device='cuda', fp16=True)
sess = ort.InferenceSession(hf_hub_download('castorini/cosdpr-distil-onnx', 'cosdpr-distil-optimized.onnx'))
texts = list(load_queries('dl19').values())[:20]
with open('data/msmarco/collection.tsv') as f:
    texts += [next(f).split('\t', 1)[1].rstrip('\n') for _ in range(20)]

ours = enc.encode(texts, max_length=256)
ref = np.stack([sess.run(['pooler_output'], {'input_ids': np.array(
    [enc.tokenizer(t, max_length=256, truncation=True)['input_ids']], dtype=np.int64)})[0][0] for t in texts])
ref /= np.linalg.norm(ref, axis=1, keepdims=True)
cos = (ours * ref).sum(1)
print(f'min cosine = {cos.min():.6f}')
assert cos.min() > 0.999, 'Encoder does not match the reference model; do not encode the corpus.'
del enc; torch.cuda.empty_cache()

## 6. Encode the corpus (resumable)

Writes `emb_XXXXX.npy` (float16) + `ids_XXXXX.txt` per 100k passages to Drive.
If the session dies, rerun cells 1–4 and this cell. If you hit CUDA out-of-memory,
lower `--batch-size`.

In [ ]:
!python -m retrievers.dpr encode --index-dir "$DPR_INDEX_DIR" --batch-size 512
!ls "$DPR_INDEX_DIR" | grep -c emb_   # expect 89

## 6b. Copy the index to local disk (before 7 and 8)

Downloads the shards from Drive through the Drive API (not the Drive mount, which
fails on long reads) to `/content/dpr_index`, and points steps 7–8 at that copy.
Resumable: files already downloaded at the right size are skipped, so rerun the
cell if it stops. Local disk is wiped when the runtime ends, so after a reconnect
run cells 1–4, then this cell, then 7/8.

In [ ]:
from google.colab import auth
auth.authenticate_user()                       # one-time Google sign-in popup
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

svc = build('drive', 'v3')

def folder_id(name, parent='root'):
    q = (f"name = '{name}' and '{parent}' in parents and trashed = false "
         "and mimeType = 'application/vnd.google-apps.folder'")
    found = svc.files().list(q=q, fields='files(id)').execute()['files']
    assert len(found) == 1, f'expected one folder named {name}, found {len(found)}'
    return found[0]['id']

fid = folder_id(os.path.basename(INDEX_DIR), folder_id(os.path.basename(DRIVE_ROOT)))
files, token = [], None
while True:
    r = svc.files().list(q=f"'{fid}' in parents and trashed = false", pageSize=1000,
                         fields='nextPageToken, files(id, name, size)', pageToken=token).execute()
    files += r['files']; token = r.get('nextPageToken')
    if not token: break
print(len(files), 'files on Drive')            # expect 179 (89 emb + 89 ids + meta.json)

LOCAL_INDEX = '/content/dpr_index'; os.makedirs(LOCAL_INDEX, exist_ok=True)
for i, f in enumerate(sorted(files, key=lambda f: f['name']), 1):
    out = os.path.join(LOCAL_INDEX, f['name'])
    if os.path.exists(out) and os.path.getsize(out) == int(f['size']):
        continue
    with io.FileIO(out + '.part', 'wb') as fh:
        dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=f['id']), chunksize=64 * 1024 * 1024)
        done = False
        while not done:
            _, done = dl.next_chunk(num_retries=5)
    os.replace(out + '.part', out)
    if i % 20 == 0: print(f'{i}/{len(files)} files')

n_emb = sum(n.startswith('emb_') and n.endswith('.npy') for n in os.listdir(LOCAL_INDEX))
print('emb files locally:', n_emb)             # expect 89
os.environ['DPR_INDEX_DIR'] = LOCAL_INDEX      # steps 7 and 8 read the local copy

## 7. Verify Stage 1 against Pyserini's published numbers

Targets (cosDPR-distil, Faiss flat): DL19 nDCG@10 0.7250 / R@1000 0.8201,
DL20 nDCG@10 0.7025 / R@1000 0.8533. Each line should say **OK** (within ±0.02).

Our run (2026-10-05): DL19 nDCG@10 0.7063 / R@1000 0.8224, DL20 0.6889 / 0.8587, all OK. The nDCG gap is not caused by fp16 or tie-breaking (see `config/dpr.yaml`).


In [ ]:
!python -m retrievers.dpr verify --index-dir "$DPR_INDEX_DIR"

## 8. Score matrices

Stage-1-only runs (baseline check via run_pipeline) and the full DPR → MiniLM
pipeline, for both query sets. Copied to Drive at the end.

If the final copy to Drive fails with `Errno 107`, remount with `drive.mount('/content/drive', force_remount=True)` and rerun the `cp` line, or download `results/scores/` from the Files panel before the runtime ends.


In [ ]:
for qs in ['dl19', 'dl20']:
    !python -m experiments.run_pipeline --retriever dpr --retriever-config config/dpr.yaml --query-set {qs} --no-reranker --output results/scores/dpr_none_{qs}.parquet
    !python -m experiments.run_pipeline --retriever dpr --retriever-config config/dpr.yaml --query-set {qs} --output results/scores/dpr_minilm_{qs}.parquet
!cp -r results/scores "$RESULTS_DIR"/ && ls -la "$RESULTS_DIR"/scores